# 06 - Interprétabilité

Ce notebook analyse les modèles de recommandation et les features contextuelles.

Nous visualisons les embeddings, mesurons l'importance des features de contexte, et examinons comment les scores de recommandation varient selon l'heure et la session.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

sns.set_theme(style='whitegrid')

sys.path.append(str((Path('..') / 'src').resolve()))
from data_utils import load_movielens_100k, encode_ids
from context_features import extract_all_context_features
from models import NCF, NCFContext

print('Python:', sys.version.split()[0])
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('matplotlib:', plt.matplotlib.__version__)
print('seaborn:', sns.__version__)


## Chargement des données et modèles

Nous rechargeons le dataset MovieLens 100K et les modèles pré-entraînés baseline et contexte.


In [ ]:
NOTEBOOK_ROOT = Path('.')
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-100k').resolve()
MODELS_DIR = (NOTEBOOK_ROOT / '..' / 'results' / 'models').resolve()
BASELINE_PATH = MODELS_DIR / 'baseline_ncf_ml100k.pt'
CONTEXT_PATH = MODELS_DIR / 'context_ncf_ml100k.pt'

df = load_movielens_100k(RAW_ROOT)
df = encode_ids(df)
df = extract_all_context_features(df)
df['label'] = (df['rating'] >= 4).astype(int)

movie_metadata = pd.read_csv(RAW_ROOT / 'movies.csv')

n_users = df['user_id_encoded'].nunique()
n_items = df['item_id_encoded'].nunique()
print('users', n_users, 'items', n_items, 'interactions', len(df))
print('baseline model exists', BASELINE_PATH.exists())
print('context model exists', CONTEXT_PATH.exists())


## Embeddings des items

Nous visualisons les embeddings d'item du modèle baseline pour voir si les films se regroupent par similarité.


In [ ]:
device = 'cpu'
baseline = NCF(num_users=n_users, num_items=n_items, embed_dim=32, mlp_layers=[64, 32, 16], dropout=0.2)
baseline.load_state_dict(torch.load(BASELINE_PATH, map_location=device))
baseline.to(device)

item_embed_gmf = baseline.embedding_item_gmf.weight.detach().cpu().numpy()
item_embed_mlp = baseline.embedding_item_mlp.weight.detach().cpu().numpy()
item_embeddings = np.concatenate([item_embed_gmf, item_embed_mlp], axis=1)

tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
item_tsne = tsne.fit_transform(item_embeddings[:2000])

movie_subset = df[['item_id', 'item_id_encoded']].drop_duplicates().merge(movie_metadata, left_on='item_id', right_on='movieId', how='left')
movie_subset = movie_subset.set_index('item_id_encoded').loc[np.arange(min(2000, n_items))]

In [ ]:
genre_counts = movie_subset['genres'].str.split('|').explode().value_counts()
top_genres = genre_counts.head(6).index.tolist()
movie_subset['primary_genre'] = movie_subset['genres'].str.split('|').str[0].where( movie_subset['genres'].str.split('|').str[0].isin(top_genres), 'Other')

plt.figure(figsize=(10, 7))
sns.scatterplot(x=item_tsne[:, 0], y=item_tsne[:, 1], hue=movie_subset['primary_genre'], palette='tab10', alpha=0.8, s=40, legend='brief')
plt.title('t-SNE des embeddings item du modèle baseline')
plt.xlabel('TSNE 1')
plt.ylabel('TSNE 2')
plt.legend(loc='best', fontsize='small')
plt.show()


## Importance des features contextuelles

Nous analysons les poids du premier layer de l'encodeur de contexte pour comprendre quels signaux sont les plus influents.


In [ ]:
context_model = NCFContext(
    num_users=n_users,
    num_items=n_items,
    context_dim=8,
    embed_dim=32,
    mlp_layers=[64, 32, 16],
    context_embed_dim=32,
    dropout=0.2,
    fusion_type='concat'
)
context_model.load_state_dict(torch.load(CONTEXT_PATH, map_location=device))
context_model.to(device)

context_weights = context_model.context_encoder.net[0].weight.detach().cpu().numpy()
feature_importance = np.sum(np.abs(context_weights), axis=0)
feature_names = [
    'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'is_weekend',
    'time_since_last_interaction_log',
    'session_length',
    'session_position_norm'
]
importance_df = pd.DataFrame({'feature': feature_names, 'importance': feature_importance})
importance_df = importance_df.sort_values('importance', ascending=False)
display(importance_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance_df, x='importance', y='feature', palette='mako')
plt.title('Importance approximative des features contextuelles')
plt.xlabel('Somme des poids absolus sur le premier layer')
plt.ylabel('Feature')
plt.show()


## Effet du contexte sur les scores

Pour un couple utilisateur/item fixe, nous comparons les scores du modèle contextuel selon différentes heures et longueurs de session.


In [ ]:
example_row = df.sample(n=1, random_state=42).iloc[0]
user_id = int(example_row['user_id_encoded'])
item_id = int(example_row['item_id_encoded'])
base_context = {
    'hour_sin': example_row['hour_sin'],
    'hour_cos': example_row['hour_cos'],
    'dow_sin': example_row['dow_sin'],
    'dow_cos': example_row['dow_cos'],
    'is_weekend': example_row['is_weekend'],
    'time_since_last_interaction_log': example_row['time_since_last_interaction_log'],
    'session_length': example_row['session_length'],
    'session_position_norm': example_row['session_position_norm']
}
variation_contexts = [
    dict(base_context, hour_sin=np.sin(2*np.pi*8/24), hour_cos=np.cos(2*np.pi*8/24)),
    dict(base_context, hour_sin=np.sin(2*np.pi*14/24), hour_cos=np.cos(2*np.pi*14/24)),
    dict(base_context, hour_sin=np.sin(2*np.pi*20/24), hour_cos=np.cos(2*np.pi*20/24)),
    dict(base_context, time_since_last_interaction_log=0.0, session_length=1, session_position_norm=1.0),
    dict(base_context, time_since_last_interaction_log=np.log1p(3600), session_length=20, session_position_norm=0.9),
]
labels = ['morning', 'afternoon', 'evening', 'fresh session', 'long session']
scores = []

for label, ctx in zip(labels, variation_contexts):
    ctx_tensor = torch.tensor([list(ctx.values())], dtype=torch.float32, device=device)
    user_tensor = torch.tensor([user_id], dtype=torch.long, device=device)
    item_tensor = torch.tensor([item_id], dtype=torch.long, device=device)
    score = context_model(user_tensor, item_tensor, ctx_tensor).item()
    scores.append({'condition': label, 'score': float(score)})

scores_df = pd.DataFrame(scores)
display(scores_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=scores_df, x='condition', y='score', palette='viridis')
plt.title('Variation des scores contextuels pour un même utilisateur/item')
plt.xlabel('Condition contextuelle')
plt.ylabel('Score brut du modèle')
plt.show()


## Conclusions d'interprétabilité

- Le t-SNE des embeddings item permet de vérifier si les films similaires se regroupent.
- L'étude des poids du premier layer de l'encodeur contextuel donne une approximation de l'importance relative de chaque feature.
- La variation des scores pour différents contextes montre comment le modèle adapte sa recommandation en fonction du comportement temporel et de session.
